<a href="https://colab.research.google.com/github/minhvu2105/alpha-evm-dex-bot/blob/main/GPT_SoVITS_Colab_VI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS v2 - Vietnamese Voice Cloning Pipeline (Optimized)

Notebook này đã được tối ưu hóa để linh hoạt và dễ sử dụng hơn.

### 🚀 Quy trình thực hiện:
1. **Cài đặt môi trường**: Chạy 1 lần đầu tiên.
2. **Cấu hình chung**: Đặt tên model (Experiment Name) tại đây.
3. **Tải Dữ liệu**: Upload file âm thanh từ máy hoặc Drive.
4. **Xử lý Dữ liệu**: Cắt âm thanh và tạo phụ đề (ASR).
5. **WebUI**: Mở giao diện để Train và Inference.

In [5]:
# @title 1. Cài đặt Môi trường, Tải Model v2Pro & Cấu hình Auto-Save Drive
import os
from google.colab import drive
from IPython.display import clear_output

# --- CẤU HÌNH DUY NHẤT ---
# @markdown Nhập tên experiment (Tên này sẽ dùng để tạo folder trên Drive):
global_exp_name = "Giong_Doc_Sach_02" # @param {type:"string"}

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
    except:
        print('Bỏ qua Mount Drive (Manual or Local Runtime)')

# 2. Clone Repo
%cd /content
if not os.path.exists("GPT-SoVITS-Vietnamese"):
    print("🚀 Đang clone repository...")
    !git clone https://github.com/tqtuan8788-ai/GPT-SoVITS-Vietnamese.git
    %cd GPT-SoVITS-Vietnamese
else:
    %cd /content/GPT-SoVITS-Vietnamese
    !git pull

# 3. Cài đặt Dependencies Hệ thống (Fix OpenCC & Build)
print("🛠️ Đang cài đặt thư viện hệ thống...")
!apt-get update && apt-get install -y libopencc-dev libsndfile1 ffmpeg cmake build-essential

# 4. Cài đặt Python Packages (Tối ưu cho GPU & v2Pro)
print("📦 Đang cài đặt danh sách thư viện tùy chỉnh...")
!pip install -q "numpy<2.0" scipy tensorboard "librosa==0.10.2" numba "pytorch-lightning>=2.4" "gradio<5" ffmpeg-python tqdm "funasr==1.0.27" cn2an pypinyin "pyopenjtalk>=0.4.1" g2p_en torchaudio modelscope sentencepiece "transformers>=4.43,<=4.50" "peft<0.18.0" chardet PyYAML psutil jieba_fast jieba split-lang "fast_langdetect>=0.3.1" wordsegment rotary_embedding_torch ToJyutping g2pk2 ko_pron python_mecab_ko "fastapi[standard]>=0.115.2" x_transformers "torchmetrics<=1.5" "pydantic<=2.10.6" "ctranslate2>=4.0,<5" "av>=11" onnxruntime-gpu
!pip install -q opencc-python-reimplemented
!pip install -q nvidia-cudnn-cu12 nvidia-cublas-cu12

# Fix CUDA PATH để ASR và Train chạy được GPU
os.environ['LD_LIBRARY_PATH'] = "/usr/local/lib/python3.10/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.10/dist-packages/nvidia/cublas/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

# 5. Tải Pretrained Models v2Pro THỦ CÔNG (ĐẦY ĐỦ)
print("📥 Đang tải trọn bộ models v2Pro...")
base_model_dir = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models"
os.makedirs(f"{base_model_dir}/v2Pro", exist_ok=True)
os.makedirs(f"{base_model_dir}/chinese-roberta-wwm-ext-large", exist_ok=True)
os.makedirs(f"{base_model_dir}/chinese-hubert-base", exist_ok=True)

# --- Tải Roberta & Hubert (Yêu cầu bắt buộc) ---
!wget -nc -P {base_model_dir}/chinese-roberta-wwm-ext-large https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/pytorch_model.bin
!wget -nc -P {base_model_dir}/chinese-roberta-wwm-ext-large https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/config.json
!wget -nc -P {base_model_dir}/chinese-roberta-wwm-ext-large https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/tokenizer.json
!wget -nc -P {base_model_dir}/chinese-hubert-base https://huggingface.co/TencentGameMate/chinese-hubert-base/resolve/main/pytorch_model.bin
!wget -nc -P {base_model_dir}/chinese-hubert-base https://huggingface.co/TencentGameMate/chinese-hubert-base/resolve/main/config.json
!wget -nc -P {base_model_dir}/chinese-hubert-base https://huggingface.co/TencentGameMate/chinese-hubert-base/resolve/main/preprocessor_config.json

# --- Tải Model v2Pro & v3 (Bản mạnh nhất hiện tại) ---
!wget -nc -P {base_model_dir}/v2Pro https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/v2Pro/s2Gv2Pro.pth
!wget -nc -P {base_model_dir}/v2Pro https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/v2Pro/s2Dv2Pro.pth
!wget -nc -P {base_model_dir} https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s1v3.ckpt

# 6. THIẾT LẬP AUTO-SAVE VÀO DRIVE (SYMBOLIC LINKS)
print("🔗 Đang kết nối trực tiếp với Google Drive...")
drive_save_path = f"/content/drive/MyDrive/GPT_SoVITS_Data/{global_exp_name}"
os.makedirs(f"{drive_save_path}/weights", exist_ok=True)
os.makedirs(f"{drive_save_path}/logs", exist_ok=True)

# Liên kết folder weights (Cứ có file .pth là bay thẳng vào Drive)
!rm -rf /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/weights
!ln -s {drive_save_path}/weights /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/weights

# Liên kết folder logs (Để WebUI nhận diện checkpoint cũ trên Drive)
!mkdir -p /content/GPT-SoVITS-Vietnamese/logs
!rm -rf /content/GPT-SoVITS-Vietnamese/logs/{global_exp_name}
!ln -s {drive_save_path}/logs /content/GPT-SoVITS-Vietnamese/logs/{global_exp_name}

clear_output()
print(f"✅ HOÀN TẤT CÀI ĐẶT!")
print(f"🎯 Experiment: {global_exp_name}")
print(f"📂 Toàn bộ model khi train xong sẽ nằm ở: Google Drive > GPT_SoVITS_Data > {global_exp_name}")

✅ HOÀN TẤT CÀI ĐẶT!
🎯 Experiment: Giong_Doc_Sach_02
📂 Toàn bộ model khi train xong sẽ nằm ở: Google Drive > GPT_SoVITS_Data > Giong_Doc_Sach_02


In [6]:
# @title ⚙️ 2. Kiểm tra Trạng thái Dữ liệu
import os

# Kiểm tra xem Bước 1 đã chạy chưa
if 'global_exp_name' not in globals():
    print("❌ Lỗi: Bạn cần chạy Bước 1 trước để thiết lập tên Experiment!")
else:
    exp_name = global_exp_name
    drive_save_path = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"

    print(f"✅ Đang làm việc với Project: {exp_name}")

    # Kiểm tra xem có model cũ không để nhắc Resume
    if os.path.exists(f"{drive_save_path}/weights"):
        files = [f for f in os.listdir(f"{drive_save_path}/weights") if f.endswith(('.pth', '.ckpt'))]
        if files:
            print(f"🔄 Tìm thấy {len(files)} file model cũ trên Drive. Bạn có thể Train tiếp (Resume).")
        else:
            print("📝 Thư mục Drive sạch sẽ, sẵn sàng Train mới.")

    # Thiết lập đường dẫn cho các bước sau
    dataset_root = "/content/dataset"
    output_root = "/content/GPT-SoVITS-Vietnamese/output"

    print(f"\n👉 Tiếp theo: Hãy upload Audio vào thư mục {dataset_root}/{exp_name}")

✅ Đang làm việc với Project: Giong_Doc_Sach_02
📝 Thư mục Drive sạch sẽ, sẵn sàng Train mới.

👉 Tiếp theo: Hãy upload Audio vào thư mục /content/dataset/Giong_Doc_Sach_02


In [7]:
# @title 3. Tải Dữ liệu Audio
# @markdown Chọn nguồn dữ liệu và upload file. Tự động dùng tên Experiment đã đặt ở Bước 1.

import os
from google.colab import files
import shutil

# 1. Kiểm tra và lấy tên Experiment
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_Default"
    print(f"⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: {exp_name}")
else:
    exp_name = global_exp_name

# 2. Thiết lập đường dẫn
# dataset_root thường được định nghĩa ở B2, nếu chưa có sẽ dùng mặc định /content/dataset
if 'dataset_root' not in globals():
    dataset_root = "/content/dataset"

input_audio_folder = f"{dataset_root}/{exp_name}"
os.makedirs(input_audio_folder, exist_ok=True)

# 3. Cấu hình nguồn tải
source_type = "Direct Upload" # @param ["Direct Upload", "Google Drive"]
# @markdown Nếu chọn Google Drive, hãy nhập đường dẫn thư mục chứa audio của bạn:
google_drive_path = "/content/drive/MyDrive/Data_Audio" # @param {type:"string"}

if source_type == "Google Drive":
    if os.path.exists(google_drive_path):
        print(f"🚀 Đang sao chép audio từ Drive: {google_drive_path}...")
        count = 0
        for f in os.listdir(google_drive_path):
            if f.lower().endswith((".wav", ".mp3", ".flac", ".m4a")):
                shutil.copy(os.path.join(google_drive_path, f), input_audio_folder)
                count += 1
        print(f"✅ Đã sao chép xong {count} file vào {input_audio_folder}")
    else:
        print(f"❌ Lỗi: Không tìm thấy thư mục '{google_drive_path}' trên Drive của bạn!")

else:
    print("📂 Vui lòng chọn các file audio từ máy tính để upload:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest_path = os.path.join(input_audio_folder, filename)
        # Nếu file đã tồn tại thì xóa trước khi di chuyển để tránh lỗi
        if os.path.exists(dest_path):
            os.remove(dest_path)
        shutil.move(filename, dest_path)
    print(f"✅ Đã upload xong vào {input_audio_folder}")

# 4. Dọn dẹp rác & Kiểm tra kết quả
# Xóa các folder ẩn hệ thống nếu có (nguyên nhân gây lỗi ASR)
junk_folder = os.path.join(input_audio_folder, ".ipynb_checkpoints")
if os.path.exists(junk_folder):
    shutil.rmtree(junk_folder)

audio_files = [f for f in os.listdir(input_audio_folder) if f.lower().endswith((".wav", ".mp3", ".flac", ".m4a"))]

print("-" * 30)
print(f"📁 Thư mục lưu trữ: {input_audio_folder}")
print(f"🎵 Tổng cộng: {len(audio_files)} file audio sẵn sàng.")
if len(audio_files) > 0:
    print(f"🔍 File tiêu biểu: {audio_files[0]}")
else:
    print("⚠️ Cảnh báo: Không tìm thấy file audio nào. Vui lòng kiểm tra lại!")

📂 Vui lòng chọn các file audio từ máy tính để upload:


Saving Giong_Doc_Sach_02.mp3 to Giong_Doc_Sach_02.mp3
✅ Đã upload xong vào /content/dataset/Giong_Doc_Sach_02
------------------------------
📁 Thư mục lưu trữ: /content/dataset/Giong_Doc_Sach_02
🎵 Tổng cộng: 1 file audio sẵn sàng.
🔍 File tiêu biểu: Giong_Doc_Sach_02.mp3


In [8]:
# @title 4. Cắt Audio + Tạo Phụ Đề Tiếng Việt (ASR)
# @markdown Tự động cắt và tạo file list dựa trên tên Experiment. Dữ liệu sẽ được lưu thẳng vào Drive.

import os
import subprocess
import sys

# 1. Kiểm tra và lấy tên Experiment
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_Default"
    print(f"⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: {exp_name}")
else:
    exp_name = global_exp_name

# 2. Thiết lập đường dẫn (Lưu vào Drive để không phải làm lại nếu sập máy)
%cd /content/GPT-SoVITS-Vietnamese
drive_base = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"
input_audio_folder = f"/content/dataset/{exp_name}"
sliced_output_folder = f"{drive_base}/sliced_audio"
asr_output_file = f"{drive_base}/{exp_name}.list"

os.makedirs(sliced_output_folder, exist_ok=True)
os.makedirs("output", exist_ok=True)

# --- FIX LỖI HỆ THỐNG ---
print("🛠️ Đang cấu hình môi trường...")
!sed -i 's|cmd=\[r"C:.*ffmpeg-win-x86_64-v7.1.exe", "-nostdin"\]|cmd=["ffmpeg", "-nostdin"]|g' tools/my_utils.py
!pip install -q faster-whisper
sys.path.insert(0, '/content/GPT-SoVITS-Vietnamese')
os.environ['PYTHONPATH'] = '/content/GPT-SoVITS-Vietnamese'

# @markdown ---
# @markdown **Tùy chọn ASR:**
use_uploaded_script = False # @param {type:"boolean"}
# @markdown Chọn 'force_re_run' nếu bạn muốn cắt lại audio từ đầu:
force_re_run = False # @param {type:"boolean"}

# ========== BƯỚC 1: CẮT AUDIO ==========
print("="*60)
print("🔪 BƯỚC 1: CẮT AUDIO THÀNH CÁC ĐOẠN NGẮN")
print("="*60)

# Kiểm tra dữ liệu đầu vào
if not os.path.exists(input_audio_folder) or len(os.listdir(input_audio_folder)) == 0:
    print(f"❌ LỖI: Thư mục {input_audio_folder} rỗng hoặc không tồn tại!")
    print("👉 Vui lòng chạy lại Bước 3 để upload audio.")
else:
    # Chỉ chạy cắt audio nếu thư mục đích trống hoặc chọn force_re_run
    if len(os.listdir(sliced_output_folder)) == 0 or force_re_run:
        print("⏳ Đang tiến hành cắt audio (Slicing)...")
        !rm -rf {sliced_output_folder}/*
        slice_cmd = f'python tools/slice_audio.py "{input_audio_folder}" "{sliced_output_folder}" -34 4000 300 10 500 0.9 0.25 0 1'
        os.system(slice_cmd)
    else:
        print(f"✅ Đã có sẵn audio đã cắt trong Drive ({len(os.listdir(sliced_output_folder))} file). Bỏ qua bước cắt.")

# ========== BƯỚC 2: ASR (NHẬN DẠNG CHỮ) ==========
print("\n" + "="*60)
if use_uploaded_script:
    print("📂 BƯỚC 2: SỬ DỤNG SCRIPT UPLOAD")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        shutil.move(filename, asr_output_file)
        print(f"✅ Đã lưu script vào: {asr_output_file}")
else:
    print("🎤 BƯỚC 2: CHẠY ASR TIẾNG VIỆT (WHISPER LARGE-V3)")
    if os.path.exists(asr_output_file) and not force_re_run:
        print(f"✅ Đã có file phụ đề (.list) trên Drive. Bỏ qua ASR.")
    else:
        print("⏳ Đang nhận diện giọng nói... (Vui lòng chờ)")
        asr_cmd = f'python tools/asr/fasterwhisper_asr.py -i "{sliced_output_folder}" -o "{drive_base}" -s large-v3 -l vi -p float16'
        os.system(asr_cmd)

        # Rename file .list cho đúng tên exp_name
        raw_list = f"{drive_base}/sliced_audio.list"
        if os.path.exists(raw_list):
            if os.path.exists(asr_output_file): os.remove(asr_output_file)
            os.rename(raw_list, asr_output_file)

# ========== TỔNG KẾT ==========
if os.path.exists(asr_output_file):
    with open(asr_output_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print("\n✅ THÀNH CÔNG!")
    print(f"📄 File Label (.list) nằm tại: {asr_output_file}")
    print(f"📝 Tổng số câu thoại: {len(lines)}")
    print(f"📁 Audio đã cắt nằm tại: {sliced_output_folder}")
    print("\n👉 BÂY GIỜ BẠN CÓ THỂ MỞ WEBUI VÀ SỬ DỤNG ĐƯỜNG DẪN .LIST TRÊN ĐỂ TRAIN.")
else:
    print("❌ LỖI: Không tìm thấy hoặc không tạo được file .list")

/content/GPT-SoVITS-Vietnamese
🛠️ Đang cấu hình môi trường...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.8 MB/s eta 0:00:00
🔪 BƯỚC 1: CẮT AUDIO THÀNH CÁC ĐOẠN NGẮN
⏳ Đang tiến hành cắt audio (Slicing)...

🎤 BƯỚC 2: CHẠY ASR TIẾNG VIỆT (WHISPER LARGE-V3)
⏳ Đang nhận diện giọng nói... (Vui lòng chờ)

✅ THÀNH CÔNG!
📄 File Label (.list) nằm tại: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/Giong_Doc_Sach_02.list
📝 Tổng số câu thoại: 29
📁 Audio đã cắt nằm tại: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/sliced_audio

👉 BÂY GIỜ BẠN CÓ THỂ MỞ WEBUI VÀ SỬ DỤNG ĐƯỜNG DẪN .LIST TRÊN ĐỂ TRAIN.


In [ ]:
# @title 5. Khởi động WebUI (Gradio)
# @markdown Cell này sẽ chạy liên tục để duy trì giao diện. Hãy click vào link `gradio.live` hiện ra ở dưới.

import os

# 1. Kiểm tra và lấy tên Experiment từ Bước 1
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_02"
    print(f"⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: {exp_name}")
else:
    exp_name = global_exp_name

%cd /content/GPT-SoVITS-Vietnamese

# 2. Cấu hình đường dẫn cho WebUI (Tự động nhận diện v2Pro)
cwd = os.getcwd()
drive_base = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"
asr_output_file = f"{drive_base}/{exp_name}.list"
sliced_folder = f"{drive_base}/sliced_audio"

# Thiết lập biến môi trường
os.environ["cnhubert_base_path"] = os.path.join(cwd, "GPT_SoVITS/pretrained_models/chinese-hubert-base")
os.environ["bert_path"] = os.path.join(cwd, "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large")
os.environ["colab_active"] = "1"
os.environ["is_share"] = "True"

# 3. Thông tin hướng dẫn nhanh
print("="*60)
print("🚀 ĐANG KHỞI ĐỘNG GPT-SOVITS WebUI...")
print("="*60)
print(f"📋 THÔNG SỐ ĐỂ ĐIỀN VÀO WEBUI (TAB 1A & 1B):")
print(f"🔹 Experiment Name: {exp_name}")
print(f"🔹 Text Label File (.list): {asr_output_file}")
print(f"🔹 Train Dataset Folder: {sliced_folder}")
print("-" * 60)
print("💡 LƯU Ý QUAN TRỌNG:")
print("1. Khi WebUI mở ra, hãy tích chọn 'v2Pro' ở ngay đầu trang.")
print("2. Mọi dữ liệu training và model (.pth, .ckpt) đã được cấu hình tự động lưu vào Drive.")
print("3. Nếu sập Runtime, chỉ cần chạy lại B1, B2 và B5 để Train tiếp.")
print("="*60)

# 4. Vá lỗi chia sẻ link và chạy WebUI
!sed -i "s/is_share = False/is_share = True/g" config.py
# Chạy với ngôn ngữ Tiếng Việt (nếu repo hỗ trợ)
!python webui.py --share

/content/GPT-SoVITS-Vietnamese
🚀 ĐANG KHỞI ĐỘNG GPT-SOVITS WebUI...
📋 THÔNG SỐ ĐỂ ĐIỀN VÀO WEBUI (TAB 1A & 1B):
🔹 Experiment Name: Giong_Doc_Sach_02
🔹 Text Label File (.list): /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/Giong_Doc_Sach_02.list
🔹 Train Dataset Folder: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/sliced_audio
------------------------------------------------------------
💡 LƯU Ý QUAN TRỌNG:
1. Khi WebUI mở ra, hãy tích chọn 'v2Pro' ở ngay đầu trang.
2. Mọi dữ liệu training và model (.pth, .ckpt) đã được cấu hình tự động lưu vào Drive.
3. Nếu sập Runtime, chỉ cần chạy lại B1, B2 và B5 để Train tiếp.
Extracting g2pw model...
Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://579de6334758bd4815.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
# @title (Tùy chọn) Chạy lại ASR thủ công
# @markdown Chỉ chạy cell này nếu bạn muốn chạy lại bước tạo phụ đề cho file audio đã cắt.

import os
if 'exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_01"

sliced_audio_folder = "output/slicer_opt"
output_list_file = f"output/{exp_name}.list"

if not os.path.exists(sliced_audio_folder):
    print("❌ LỖI: Không tìm thấy thư mục đã cắt!")
else:
    print("🚀 Đang chạy Whisper (large-v3)...")
    !python tools/asr/fasterwhisper_asr.py -i "{sliced_audio_folder}" -o "output" -s large-v3 -l vi -p float16

    # Rename
    generated = "output/slicer_opt.list"
    if os.path.exists(generated):
        import shutil
        shutil.move(generated, output_list_file)

    print(f"✅ Đã xong: {os.path.abspath(output_list_file)}")

In [ ]:
# @title 6. Kiểm tra & Lưu trữ Model Vĩnh viễn (Final Sync)
# @markdown Cell này giúp bạn kiểm tra lại các file đã train và đảm bảo chúng an toàn trên Drive.

import os
import shutil

# 1. Lấy tên Experiment
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_02"
else:
    exp_name = global_exp_name

# 2. Thiết lập đường dẫn lưu trữ
# Chúng ta lưu vào folder Weights riêng để sau này dễ tìm kiếm
final_drive_path = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}/FINAL_MODELS"
os.makedirs(final_drive_path, exist_ok=True)

print(f"🕵️ Đang kiểm tra thành quả của Experiment: {exp_name}...")

# 3. Gom các file weights (.pth và .ckpt)
# Vì đã dùng link ảo nên weights_dir thực tế trỏ về Drive
weights_dir = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/weights"

if os.path.exists(weights_dir):
    trained_files = [f for f in os.listdir(weights_dir) if f.endswith(('.pth', '.ckpt'))]

    if len(trained_files) > 0:
        print(f"✅ Tìm thấy {len(trained_files)} file model đã huấn luyện.")

        # Sao chép thêm một bản vào thư mục FINAL_MODELS cho chắc chắn
        for file in trained_files:
            src = os.path.join(weights_dir, file)
            dst = os.path.join(final_drive_path, file)
            shutil.copy2(src, dst)
            print(f"   -> Đã sao lưu: {file}")

        print(f"\n🎉 CHÚC MỪNG! Model của bạn đã được bảo vệ vĩnh viễn tại:")
        print(f"📂 {final_drive_path}")
    else:
        print("⚠️ Cảnh báo: Không tìm thấy file model mới nào trong thư mục weights.")
        print("👉 Hãy đảm bảo bạn đã nhấn nút 'Start Training' trong WebUI và đã hoàn thành ít nhất 1 Epoch.")
else:
    print("❌ Lỗi: Thư mục weights không tồn tại.")

# 4. Hiển thị dung lượng đã sử dụng trên Drive (Tùy chọn)
!du -sh "{final_drive_path}"